# Phase 2 — Image Processing and Rectification

Goals:

- learn the OpenCV image representation and coordinate system;
- rectify MIDV photographs using ground-truth `doc_quad`;
- normalize `alb_id` into a canonical image size;
- map template field geometry into canonical coordinates;
- crop the seven required fields;
- implement and calibrate controlled image degradations.

In [ ]:
from pathlib import Path
import json

import cv2
import numpy as np
import matplotlib.pyplot as plt

Let's use the same doctype and image from the previous notebook.

In [ ]:
current_directory = Path.cwd().resolve()
if (current_directory / "PROJECT.md").exists():
    repository_root = current_directory
elif (current_directory.parent / "PROJECT.md").exists():
    repository_root = current_directory.parent
else:
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks directory.")

DATASET_ROOT = repository_root / "data/raw"

DOC_TYPE = "alb_id"
SAMPLE_ID = "00.jpg"

template_image_path = DATASET_ROOT / "templates/images" / DOC_TYPE / SAMPLE_ID
photo_image_path = DATASET_ROOT / "photo/images" / DOC_TYPE / SAMPLE_ID

if not photo_image_path.is_file():
    raise FileNotFoundError(photo_image_path)



Now let's load the photograph with OpenCV

In [ ]:
image_bgr = cv2.imread(str(photo_image_path))

assert image_bgr is not None

print(type(image_bgr))
print(image_bgr.dtype)
print(image_bgr.shape)

`cv2.imread()` does not return some special `Image` class. You get a regular NumPy array.

For `image_bgr.shape` the convention is `(height, width, channels)`.

That's really important to understand now, a point `(x,y)` means:
```
x = horizontal position = column
y = vertical position = row
```
But direct NumPy indexing means `image[y,x]`

Let's inspect one pixel:

In [ ]:
y = 400
x = 200

pixel = image_bgr[y, x]

print(pixel)

One more caveat: `OpenCV's` default channel order is B,G,R rather than R,G,B.

Look what happens when we display the image in matplotlib (which uses R,G,B ordering).

In [ ]:
plt.figure(figsize=(8, 8))
plt.imshow(image_bgr)
plt.axis("off")
plt.show()

Weird colors! Let's fix it.

In [ ]:
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(8, 8))
plt.imshow(image_rgb)
plt.axis("off")
plt.show()

Back to normal.

In [ ]:
print(image_bgr.dtype)
print(image_bgr.shape)
print(photo_image_path)

## Coordinates and Basic Image Operations

Let's make the image's dimensions explicit

In [ ]:
height, width = image_bgr.shape[:2]

print("width:", width)
print("height:", height)

Let's put a marker exactly at the center of the photograph

In [ ]:
center_x = width // 2
center_y = height // 2

debug_image = image_bgr.copy()

cv2.circle(
    debug_image,
    (center_x, center_y),
    radius=30,
    color=(0, 0, 255),
    thickness=-1,
)

plt.figure(figsize=(8, 10))
plt.imshow(cv2.cvtColor(debug_image, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

Notice the `copy()`.

Functions such as:
```python
cv2.circle()
cv2.rectangle()
cv2.line()
```
Modify the NumPy array you give them in place. If we drew directly onto `image_bgr`, we'd be modifying our working input.

### Grayscale


In [ ]:
gray_image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

print(gray_image.shape)
print(gray_image.dtype)

plt.figure(figsize=(8, 10))
plt.imshow(gray_image, cmap="gray")
plt.axis("off")
plt.show()

We've gone from three values per pixel to one intensity, roughly representing brightness.

This matters because quite a few classical CV algorithms don't actually need color. Later things such as edge detection, Laplacian sharpness measurement, thresholding, and contour detection will commonly operate on grayscale images.

For our project, one of the router features we've already specified is variance of the grayscale Laplacian as a cheap sharpness measurement, so this isn't just an OpenCV tutorial detour—it returns later in the actual experiment.

### Resizing

In [ ]:
small_image = cv2.resize(
    image_bgr,
    (width // 4, height // 4),
)

print("original:", image_bgr.shape)
print("resized:", small_image.shape)

`cv2.resize()` expects (width, height). While NumPy shape reports (height, width, channels). So, `cv2.resize(image, (567,1008))` produces `shape = (1008, 567, 3)`.

Now look at the size difference:


In [ ]:
plt.figure(figsize=(6, 8))
plt.imshow(cv2.cvtColor(small_image, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

### Downscale Degradation
Look what happens when we agressively resize the image down and then enlarge it back:

In [ ]:
tiny = cv2.resize(
    image_bgr,
    (width // 10, height // 10),
    interpolation=cv2.INTER_AREA,
)

restored = cv2.resize(
    tiny,
    (width, height),
    interpolation=cv2.INTER_LINEAR,
)

plt.figure(figsize=(8, 10))
plt.imshow(cv2.cvtColor(restored, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

The final array once again contains millions of pixels, but the information destroyed during downsampling is gone.

## Document quadrilateral

MIDV photo annotations contain a `doc_quad` polygon describing the four
corners of the physical document in the photograph.

Before performing perspective rectification, we will:

1. retrieve the polygon from the VIA annotation;
2. represent its vertices as a NumPy array;
3. draw and label the vertices;
4. inspect the geometry of its four edges.

Let's inspect the photo regions. (I've done a more detailed investigation on notebook one!)

In [ ]:
photo_annotation_path = DATASET_ROOT / "photo" / "annotations" / "alb_id.json"

with open(photo_annotation_path, "r", encoding="utf-8") as f:
    photo_annotations = json.load(f)

metadata = photo_annotations["_via_img_metadata"]

filename = photo_image_path.name

photo_record = None

for record in metadata.values():
    if record["filename"] == filename:
        photo_record = record
        break

for region in photo_record["regions"]:
    print(region["region_attributes"])

We will draw the same doc_quad region from the previous notebook, but this time using `openCV`.

In [ ]:
doc_quad_region = None

for region in photo_record["regions"]:
    field_name = region["region_attributes"].get("field_name")

    if field_name == "doc_quad":
        doc_quad_region = region
        break

doc_quad_shape = doc_quad_region["shape_attributes"]

x_coordinates = doc_quad_shape["all_points_x"]
y_coordinates = doc_quad_shape["all_points_y"]

document_corners = np.array(
    list(zip(x_coordinates, y_coordinates)),
    dtype=np.float32,
)

print(document_corners)
print(document_corners.shape)
print(document_corners.dtype)

Drawing the actual quadrilateral

In [ ]:
annotated_photo = image_bgr.copy()

corners_int = document_corners.astype(np.int32) # OpenCV drawing functions generally prefer Integer coordinates

For Polygons, OpenCV expects and extra dimension.

In [ ]:
polygon = corners_int.reshape((-1, 1, 2))

print(corners_int.shape)
print(polygon.shape)

In [ ]:
cv2.polylines(
    annotated_photo,
    [polygon],
    isClosed=True,
    color=(0, 255, 0),
    thickness=10,
)

plt.figure(figsize=(8, 10))
plt.imshow(cv2.cvtColor(annotated_photo, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

Let's inspect the distances between those points.

In [ ]:
top_width = np.linalg.norm(document_corners[1] - document_corners[0])
bottom_width = np.linalg.norm(document_corners[2] - document_corners[3])

left_height = np.linalg.norm(document_corners[3] - document_corners[0])
right_height = np.linalg.norm(document_corners[2] - document_corners[1])

print(f"top width:     {top_width:.2f}")
print(f"bottom width:  {bottom_width:.2f}")
print(f"left height:   {left_height:.2f}")
print(f"right height:  {right_height:.2f}")

`document_corners[1] - document_corners[0]` computes the vector $(x_1 - x_0, y_1 - y_0)$

And `np.linalg.norm(...)` computes its euclidean length:
$\sqrt{(x_1 - x_0)^2 + (y_1 - y_0)^2}$


Notice that the top width != bottom width. That's perspective. And we're going to be solving that with rectification. The goal is:
> Find a transformation that maps these four known source points onto the four corners of a rectangle.

# Transformations
Let's start modeling and altering the image as we wish. Power!!

We'll start using the native dimensions of the corresponding clean template, preserving the known `alb_id` aspect ratio.

Let's start by defining the destination points (the flat coordinate system we want) from the template shape.

In [ ]:
template_bgr = cv2.imread(str(template_image_path))

assert template_bgr is not None, f"Could not load {template_image_path}"

template_height, template_width = template_bgr.shape[:2]

destination_corners = np.array(
    [
        [0, 0],
        [template_width - 1, 0],
        [template_width - 1, template_height - 1],
        [0, template_height - 1],
    ],
    dtype=np.float32,
)

print("Source:")
print(document_corners)

print("\nDestination:")
print(destination_corners)


Ordering is deliberately identical here. (ex: index 0 -> top-left, index 3 -> bottom-left).

This correspondence is essential. If we don't consider this, our transformation will obediently compute a transformation that twists the image in a weird way.


## The Transformation

We're looking for a *homography*

For a point in the photograph:

$$(x, y)$$

we want a corresponding point:

$$(x', y')$$

in the rectified card.

A perspective transformation is represented by a $3 \times 3$ matrix:

$$H = \begin{bmatrix} h_{11} & h_{12} & h_{13} \\ h_{21} & h_{22} & h_{23} \\ h_{31} & h_{32} & h_{33} \end{bmatrix}$$

Instead of ordinary 2D coordinates, projective geometry writes the point as:

$$\begin{bmatrix} x \\ y \\ 1 \end{bmatrix}$$

and transforms it with:

$$\begin{bmatrix} u \\ v \\ w \end{bmatrix} = H \begin{bmatrix} x \\ y \\ 1 \end{bmatrix}$$

Then the actual output coordinates are:

$$x' = \frac{u}{w}, \qquad y' = \frac{v}{w}$$

That division by w is what gives a homography the ability to represent *perspective*, not merely translation/rotation/scaling.

Therefore, the transformation can be described as:
> Given four source points and four corresponding destination points, find the perspective transformation that maps one set onto the other.

In OpenCV, `getPerspectiveTransform()` is the one who does it:

In [ ]:
homography = cv2.getPerspectiveTransform(
    document_corners,
    destination_corners,
)

print(homography)
print(homography.shape)

Numbers are weird here. That's okay.

### Verify the transformation before warping the image

Useful Habit! As we mentioned: `document_corners` should map into `destination_corners`, so let's actually test that.


In [ ]:
transformed_corners = cv2.perspectiveTransform(
    document_corners.reshape(-1, 1, 2),
    homography,
).reshape(-1, 2)

print("Expected destination corners:")
print(destination_corners)

print("\nActual transformed corners:")
print(transformed_corners)

Now, we're ready to warp the pixels!

In [ ]:
rectified_bgr = cv2.warpPerspective(
    image_bgr,
    homography,
    (template_width, template_height),
)

print(rectified_bgr.shape)

Time to see if it worked.

In [ ]:
plt.figure(figsize=(12, 8))

plt.imshow(cv2.cvtColor(rectified_bgr, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Rectified MIDV photograph")

plt.show()

Let's compare it against the clean template to see why using the template dimensions was useful.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].imshow(cv2.cvtColor(template_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title("Clean template")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(rectified_bgr, cv2.COLOR_BGR2RGB))
axes[1].set_title("Rectified photograph")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## Using template annotations to crop a real field.
Now it's time to connect the two annotation worlds. Because we rectified the photograph to exactly the template's width and height, the template field coordinates can now be used directly on `rectified_bgr`. That is exactly the simplification intended for the v1 OCR pipeline.

We'll start with one field only: `surname`. Once we've verified the geometry, generalizing to all seven fields will be straightforward.

In [ ]:
template_annotation_path = (
    DATASET_ROOT / "templates" / "annotations" / "alb_id.json"
)

with open(template_annotation_path, "r", encoding="utf-8") as f:
    template_annotations = json.load(f)

template_metadata = template_annotations["_via_img_metadata"]

print(len(template_metadata))

Loaded! Let's get the annotation related to the image we're using

In [ ]:
template_filename = template_image_path.name

template_record = None

for record in template_metadata.values():
    if record["filename"] == template_filename:
        template_record = record
        break

assert template_record is not None, (
    f"No template annotation found for {template_filename}"
)

print(template_record["filename"])
print(len(template_record["regions"]))

Now let's find the surname annotation. Unlike `doc_quad`, which was a polygon, template fields are rectangular sections.

In [ ]:
surname_region = None

for region in template_record["regions"]:
    field_name = region["region_attributes"].get("field_name")

    if field_name == "surname":
        surname_region = region
        break

print(surname_region["region_attributes"])
print(surname_region["shape_attributes"])

Now let's extract the rectangle's coordinates

In [ ]:
surname_shape = surname_region["shape_attributes"]

x = surname_shape["x"]
y = surname_shape["y"]
width = surname_shape["width"]
height = surname_shape["height"]


x1 = x
y1 = y
x2 = x + width
y2 = y + height

print(f"x1={x1}, y1={y1}")
print(f"x2={x2}, y2={y2}")

print("ground truth:", surname_region["region_attributes"]["value"])

Before cropping anything, let's once again draw the surname box on the clean template.

In [ ]:
template_debug = template_bgr.copy()

cv2.rectangle(
    template_debug,
    (x1, y1),
    (x2, y2),
    color=(0, 0, 255),
    thickness=5,
)

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(template_debug, cv2.COLOR_BGR2RGB))
plt.title("Surname annotation on clean template")
plt.axis("off")
plt.show()

Now we take exactly the same rectangle and draw it in the rectified photograph.

In [ ]:
rectified_debug = rectified_bgr.copy()

cv2.rectangle(
    rectified_debug,
    (x1, y1),
    (x2, y2),
    color=(0, 0, 255),
    thickness=5,
)

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(rectified_debug, cv2.COLOR_BGR2RGB))
plt.title("Same surname coordinates on rectified photograph")
plt.axis("off")
plt.show()

The rectification should surround the surname in the photograph too.

### Cropping the surname
Let's use plain NumPy slicing

In [ ]:
template_surname_crop = template_bgr[y1:y2, x1:x2]
rectified_surname_crop = rectified_bgr[y1:y2, x1:x2]

fig, axes = plt.subplots(2,1,figsize=(12,4))

axes[0].imshow(
    cv2.cvtColor(template_surname_crop, cv2.COLOR_BGR2RGB)
)
axes[0].set_title("Surname - Clean Template")
axes[0].axis("off")

axes[1].imshow(
    cv2.cvtColor(rectified_surname_crop, cv2.COLOR_BGR2RGB)
)
axes[1].set_title("Surname - Rectified Photograph")
axes[1].axis("off")

plt.tight_layout()
plt.show()

Wait....the rectified cropped image is a bit off.

That's why we should add some padding:

In [ ]:
padding_x = int(width * 0.10)
padding_y = int(height * 0.15)

crop_x1 = max(0, x1 - padding_x)
crop_y1 = max(0, y1 - padding_y)

crop_x2 = min(template_width, x2 + padding_x)
crop_y2 = min(template_height, y2 + padding_y)

rectified_surname_crop_padded = rectified_bgr[crop_y1:crop_y2, crop_x1:crop_x2]

plt.figure(figsize=(12, 3))
plt.imshow(
    cv2.cvtColor(rectified_surname_crop_padded, cv2.COLOR_BGR2RGB)
)
plt.axis("off")
plt.show()


As a final practice, let's to the same with the `birth_date` field.

In [ ]:
birth_date_region = None

for region in template_record["regions"]:
    if region["region_attributes"].get("field_name") == "birth_date":
        birth_date_region = region
        break

birth_date_shape = birth_date_region["shape_attributes"]

bd_x1 = birth_date_shape["x"]
bd_y1 = birth_date_shape["y"]

bd_x2 = bd_x1 + birth_date_shape["width"]
bd_y2 = bd_y1 + birth_date_shape["height"]

template_birth_date_crop = template_bgr[bd_y1:bd_y2, bd_x1:bd_x2]
rectified_birth_date_crop = rectified_bgr[bd_y1:bd_y2, bd_x1:bd_x2]

fig, axes = plt.subplots(2,1,figsize=(12,4))

axes[0].imshow(
    cv2.cvtColor(template_birth_date_crop, cv2.COLOR_BGR2RGB)
)
axes[0].set_title("Birth Date - Clean Template")
axes[0].axis("off")


axes[1].imshow(
    cv2.cvtColor(rectified_birth_date_crop, cv2.COLOR_BGR2RGB)
)
axes[1].set_title("Birth Date - Rectified Photograph")
axes[1].axis("off")

plt.tight_layout()
plt.show()

And for the `number` field:

In [ ]:
birth_date_region = None

for region in template_record["regions"]:
    if region["region_attributes"].get("field_name") == "number":
        birth_date_region = region
        break

birth_date_shape = birth_date_region["shape_attributes"]

bd_x1 = birth_date_shape["x"]
bd_y1 = birth_date_shape["y"]

bd_x2 = bd_x1 + birth_date_shape["width"]
bd_y2 = bd_y1 + birth_date_shape["height"]

template_birth_date_crop = template_bgr[bd_y1:bd_y2, bd_x1:bd_x2]
rectified_birth_date_crop = rectified_bgr[bd_y1:bd_y2, bd_x1:bd_x2]

fig, axes = plt.subplots(2,1,figsize=(12,4))

axes[0].imshow(
    cv2.cvtColor(template_birth_date_crop, cv2.COLOR_BGR2RGB)
)
axes[0].set_title("Birth Date - Clean Template")
axes[0].axis("off")


axes[1].imshow(
    cv2.cvtColor(rectified_birth_date_crop, cv2.COLOR_BGR2RGB)
)
axes[1].set_title("Birth Date - Rectified Photograph")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## Final Validation - Round Trip
Final part now is going to be validating the rectification on several IDs before building the reusable field-cropping layer.

Let's turn rectification in into a function and test several identities.

In [ ]:
def rectify_document(image_bgr, document_corners, output_width, output_height):
    destination_corners = np.array(
        [
            [0, 0],
            [output_width - 1, 0],
            [output_width - 1, output_height - 1],
            [0, output_height - 1],
        ],
        dtype=np.float32,
    )

    homography = cv2.getPerspectiveTransform(
        document_corners.astype(np.float32),
        destination_corners,
    )

    rectified = cv2.warpPerspective(
        image_bgr,
        homography,
        (output_width, output_height),
    )

    return rectified

This is basically what Phase 2 requires as a deliverable!

Let's now load the annotation JSON once.

In [ ]:
photo_annotation_path = (
    DATASET_ROOT / "photo" / "annotations" / "alb_id.json"
)

with open(photo_annotation_path, "r", encoding="utf-8") as f:
    photo_annotations = json.load(f)

photo_metadata = photo_annotations["_via_img_metadata"]

Now add two small helpers:

In [ ]:
def find_record(metadata, filename):
    for record in metadata.values():
        if record["filename"] == filename:
            return record

    raise ValueError(f"No annotation found for {filename}")

def extract_document_corners(record):
    for region in record["regions"]:

        attributes = region["region_attributes"]

        if attributes.get("field_name") == "doc_quad":

            shape = region["shape_attributes"]

            return np.array(
                list(
                    zip(
                        shape["all_points_x"],
                        shape["all_points_y"],
                    )
                ),
                dtype=np.float32,
            )
    raise ValueError("No doc_quad found.")

Let's keep a tiny sample.

In [ ]:
sample_ids = ["00", "01", "02", "03", "04"]


fig, axes = plt.subplots(
    len(sample_ids),
    3,
    figsize = (15, 5 * len(sample_ids))
)

for row, sample_id in enumerate(sample_ids):
    filename = f"{sample_id}.jpg"

    photo_path = (
        DATASET_ROOT / "photo"
        / "images"
        / "alb_id"
        / filename
    )

    template_path = (
        DATASET_ROOT
        / "templates"
        / "images"
        / "alb_id"
        / filename
    )

    photo_bgr = cv2.imread(str(photo_path))
    template_bgr_sample = cv2.imread(str(template_path))

    template_height_sample , template_width_sample = template_bgr_sample.shape[:2]

    record = find_record(photo_metadata, filename)
    corners = extract_document_corners(record)

    rectified = rectify_document(photo_bgr,
                                 corners,
                                 template_width_sample,
                                 template_height_sample
                                 )

    template_rgb = cv2.cvtColor(template_bgr_sample, cv2.COLOR_BGR2RGB)

    rectified_rgb = cv2.cvtColor(rectified, cv2.COLOR_BGR2RGB)

    overlay = cv2.addWeighted(
        template_rgb,
        0.5,
        rectified_rgb,
        0.5,
        0
    )

    axes[row,0].imshow(template_rgb)
    axes[row,0].set_title(f"Sample {sample_id} - Template")

    axes[row,1].imshow(rectified_rgb)
    axes[row,1].set_title(f"Sample {sample_id} - Rectified")

    axes[row,2].imshow(overlay)
    axes[row,2].set_title(f"Sample {sample_id} - Overlay")

    for axis in axes[row]:
        axis.axis("off")


plt.tight_layout()
plt.show()



Let's do some testing for padding.

In [ ]:
template_annotation_path = (
    DATASET_ROOT
    / "templates"
    / "annotations"
    / "alb_id.json"
)

with open(template_annotation_path, "r", encoding="utf-8") as f:
    template_annotations = json.load(f)

template_metadata = template_annotations["_via_img_metadata"]

Tiny Helper for finding one field region:

In [ ]:
def find_field_region(record, field_name):
    for region in record["regions"]:
        attributes = region["region_attributes"]

        if attributes.get("field_name") == field_name:
            return region

    raise ValueError(f"No field named {field_name}")

Let's compare tight vs padded surname crops:

In [ ]:
padding_x = 0.10
vertical_paddings = [0.20, 0.30, 0.40]

In [ ]:
fig, axes = plt.subplots(
    len(sample_ids),
    len(vertical_paddings),
    figsize=(15, 3 * len(sample_ids)),
)

for row, sample_id in enumerate(sample_ids):
    filename = f"{sample_id}.jpg"

    photo_path = (
        DATASET_ROOT
        / "photo"
        / "images"
        / "alb_id"
        / filename
    )

    template_path = (
        DATASET_ROOT
        / "templates"
        / "images"
        / "alb_id"
        / filename
    )

    photo_bgr = cv2.imread(str(photo_path))
    template_bgr_sample = cv2.imread(str(template_path))

    template_height_sample, template_width_sample = (
        template_bgr_sample.shape[:2]
    )

    photo_record = find_record(photo_metadata, filename)
    corners = extract_document_corners(photo_record)

    rectified = rectify_document(
        photo_bgr,
        corners,
        template_width_sample,
        template_height_sample,
    )

    template_record = find_record(
        template_metadata,
        filename,
    )

    surname_region = find_field_region(
        template_record,
        "surname",
    )

    shape = surname_region["shape_attributes"]

    x = shape["x"]
    y = shape["y"]
    width = shape["width"]
    height = shape["height"]

    for col, padding_y in enumerate(vertical_paddings):
        pad_x = int(width * 0.10)
        pad_y = int(height * padding_y)

        x1 = max(0, x - pad_x)
        y1 = max(0, y - pad_y)

        x2 = min(
            template_width_sample,
            x + width + pad_x,
        )

        y2 = min(
            template_height_sample,
            y + height + pad_y,
        )

        crop = rectified[y1:y2, x1:x2]

        axes[row, col].imshow(
            cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        )

        axes[row, col].set_title(
            f"{sample_id} — y padding {padding_y:.0%}"
        )

        axes[row, col].axis("off")

plt.tight_layout()
plt.show()

Let's now crop all seven required fields:

In [ ]:
required_fields = [
    "number",
    "surname",
    "name",
    "birth_date",
    "expiry_date",
    "nationality",
    "gender",
]

In [ ]:
def crop_field(
    image,
    shape,
    padding_x=0.10,
    padding_top=0.10,
    padding_bottom=0.40,
):
    x = shape["x"]
    y = shape["y"]
    width = shape["width"]
    height = shape["height"]

    image_height, image_width = image.shape[:2]

    pad_x = int(width * padding_x)
    pad_top = int(height * padding_top)
    pad_bottom = int(height * padding_bottom)

    x1 = max(0, x - pad_x)
    y1 = max(0, y - pad_top)

    x2 = min(
        image_width,
        x + width + pad_x,
    )

    y2 = min(
        image_height,
        y + height + pad_bottom,
    )

    return image[y1:y2, x1:x2]

In [ ]:
field_crops = {}

for field_name in required_fields:

    region = find_field_region(
        template_record,
        field_name
    )

    crop = crop_field(
        rectified_bgr,
        region["shape_attributes"]
    )
    field_crops[field_name] = crop



fig, axes = plt.subplots(
    len(required_fields),
    1,
    figsize=(12, 2.5 * len(required_fields)),
)

for axis, field_name in zip(axes, required_fields):
    crop = field_crops[field_name]

    axis.imshow(
        cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    )

    axis.set_title(field_name)
    axis.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(
    len(sample_ids),
    len(required_fields),
    figsize=(4 * len(required_fields), 3 * len(sample_ids)),
)

for row, sample_id in enumerate(sample_ids):
    filename = f"{sample_id}.jpg"

    photo_path = (
        DATASET_ROOT
        / "photo"
        / "images"
        / "alb_id"
        / filename
    )

    template_path = (
        DATASET_ROOT
        / "templates"
        / "images"
        / "alb_id"
        / filename
    )

    photo_bgr = cv2.imread(str(photo_path))
    template_bgr_sample = cv2.imread(str(template_path))

    assert photo_bgr is not None
    assert template_bgr_sample is not None

    template_height_sample, template_width_sample = (
        template_bgr_sample.shape[:2]
    )

    # Photo annotation -> document corners
    photo_record = find_record(
        photo_metadata,
        filename,
    )

    corners = extract_document_corners(
        photo_record
    )

    # Rectify photograph
    rectified = rectify_document(
        photo_bgr,
        corners,
        template_width_sample,
        template_height_sample,
    )

    # Template annotation -> field rectangles
    template_record = find_record(
        template_metadata,
        filename,
    )

    for col, field_name in enumerate(required_fields):
        region = find_field_region(
            template_record,
            field_name,
        )

        crop = crop_field(
            rectified,
            region["shape_attributes"],
        )

        axes[row, col].imshow(
            cv2.cvtColor(
                crop,
                cv2.COLOR_BGR2RGB,
            )
        )

        if row == 0:
            axes[row, col].set_title(field_name)

        if col == 0:
            axes[row, col].set_ylabel(
                sample_id,
                fontsize=12,
            )

        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])

plt.tight_layout()
plt.show()

The padded template-coordinate crop strategy worked for nearly all inspected
fields across five identities. One `gender` crop partially clipped the target
character, suggesting that very small fields may need a minimum absolute
padding in addition to percentage-based padding.

### Exploring Canonical Resolution

How small can we make the canonical card while keeping the smallest required fields readable?

Let's start by comparing a few scale factors:

In [ ]:
candidate_scales = [
    1.00,
    0.75,
    0.50,
    0.35,
]

In [ ]:
candidate_sizes = []

for scale in candidate_scales:
    width = round(template_width * scale)
    height = round(template_height * scale)

    candidate_sizes.append(
        (scale, width, height)
    )

candidate_sizes

Let's now resize the canonical card into those candidate sizes:

In [ ]:
resized_candidates = {}

for scale, width, height in candidate_sizes:
    resized = cv2.resize(
        rectified_bgr,
        (width, height),
        interpolation=cv2.INTER_AREA,
    )

    resized_candidates[scale] = resized

Now let's visualize them:

In [ ]:
fix, axes = plt.subplots(
    len(candidate_scales),
    1,
    figsize=(12,5*len(candidate_scales))
)

for axis, scale in zip(axes, candidate_scales):
    image = resized_candidates[scale]

    axis.imshow(
        cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    )

    axis.set_title(
        f"{scale:.0%} — {image.shape[1]} × {image.shape[0]}"
    )

    axis.axis("off")

plt.tight_layout()
plt.show()

It's essential to also scale the field coordinates.

Let's use `gender`, because it already exposed the weakness of small fields.


In [ ]:
gender_region = find_field_region(
    template_record,
    "gender",
)

gender_shape = gender_region["shape_attributes"]

Now crop it at each resolution:

In [ ]:
fig, axes = plt.subplots(
    len(candidate_scales),
    1,
    figsize=(10, 2.5 * len(candidate_scales)),
)

for axis, scale in zip(axes, candidate_scales):
    image = resized_candidates[scale]

    x = round(gender_shape["x"] * scale)
    y = round(gender_shape["y"] * scale)

    width = round(gender_shape["width"] * scale)
    height = round(gender_shape["height"] * scale)

    # Use the same working padding policy.
    pad_x = round(width * 0.10)
    pad_top = round(height * 0.10)
    pad_bottom = round(height * 0.40)

    image_height, image_width = image.shape[:2]

    x1 = max(0, x - pad_x)
    y1 = max(0, y - pad_top)

    x2 = min(
        image_width,
        x + width + pad_x,
    )

    y2 = min(
        image_height,
        y + height + pad_bottom,
    )

    crop = image[y1:y2, x1:x2]

    axis.imshow(
        cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    )

    axis.set_title(
        f"{scale:.0%} — crop shape {crop.shape[:2]}"
    )

    axis.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
for scale, _, _ in candidate_sizes:
    field_width = round(
        gender_shape["width"] * scale
    )

    field_height = round(
        gender_shape["height"] * scale
    )

    print(
        f"{scale:.0%}: "
        f"{field_width} × {field_height} px"
    )

### Canonical resolution candidate

The native `alb_id` template size is 2167 × 1360.

Candidate scales of 100%, 75%, 50%, and 35% were visually inspected,
with particular attention to the small `gender` field.

At 50%, the `gender` annotation remains approximately 26 × 24 pixels
and the character remains clearly legible, while image area is reduced
substantially.

The canonical resolution selected for v1 is:

**1084 × 680 pixels (50% of the native template dimensions).**

The value is frozen in `configs/controlled_degradations_v1.json`. The final helpers below use that file, and the training/validation geometry audit is summarized in `docs/phases/phase-2.md`.

## Synthesizing

Now it's time to make preprocessing reusable based on our investigation:


The target should be something like:

with open(repository_root / "configs/controlled_degradations_v1.json", encoding="utf-8") as f:
    phase2_config = json.load(f)
CANONICAL_WIDTH = phase2_config["canonical_width"]
CANONICAL_HEIGHT = phase2_config["canonical_height"]

find_record(...)
extract_document_corners(...)
find_field_region(...)
rectify_document(...)
scale_field_shape(...)
crop_field(...)

And then the actual preprocessing of one sample should read cleanly, roughly:

```python
photo_record = find_record(photo_metadata, filename)
template_record = find_record(template_metadata, filename)

corners = extract_document_corners(photo_record)

canonical_image = rectify_document(
    photo_bgr,
    corners,
    CANONICAL_WIDTH,
    CANONICAL_HEIGHT,
)

region = find_field_region(template_record, "surname")

canonical_shape = scale_field_shape(
    region["shape_attributes"],
    template_width,
    template_height,
    CANONICAL_WIDTH,
    CANONICAL_HEIGHT,
)

surname_crop = crop_field(
    canonical_image,
    canonical_shape,
)
```

## Reusable preprocessing pipeline

The previous sections established that:

- MIDV `doc_quad` annotations can rectify photographs reliably;
- a canonical resolution of 1084 × 680 is a reasonable working candidate;
- template field coordinates can be mapped into canonical coordinates;
- padded field crops tolerate the small registration differences between
  templates and rectified photographs.

This section consolidates those experiments into small reusable functions.
The canonical size and degradation parameters are frozen in the versioned configuration. The final crop helper adds a minimum horizontal margin after the small-field inspection above.

In [ ]:
CANONICAL_WIDTH = 1084
CANONICAL_HEIGHT = 680

FIELD_PADDING_X = 0.10
FIELD_MIN_PADDING_X = 12
FIELD_PADDING_TOP = 0.10
FIELD_PADDING_BOTTOM = 0.40

REQUIRED_FIELDS = [
    "number",
    "surname",
    "name",
    "birth_date",
    "expiry_date",
    "nationality",
    "gender",
]

### VIA Lookup Helpers


In [ ]:
def find_record(metadata, filename):
    for record in metadata.values():
        if record["filename"] == filename:
            return record

    raise ValueError(f"No annotation found for {filename}")

In [ ]:
def find_field_region(record, field_name):
    for region in record["regions"]:
        attributes = region["region_attributes"]

        if attributes.get("field_name") == field_name:
            return region

    raise ValueError(f"No field named {field_name}")

In [ ]:
def extract_document_corners(record):
    region = find_field_region(record, "doc_quad")
    shape = region["shape_attributes"]

    return np.array(
        list(
            zip(
                shape["all_points_x"],
                shape["all_points_y"],
            )
        ),
        dtype=np.float32,
    )

All this code means:

record -> find `doc_quad` region -> convert polygon to four points

### Rectify Document Method

In [ ]:
def rectify_document(
    image,
    document_corners,
    output_width=CANONICAL_WIDTH,
    output_height=CANONICAL_HEIGHT,
):
    destination_corners = np.array(
        [
            [0, 0],
            [output_width - 1, 0],
            [output_width - 1, output_height - 1],
            [0, output_height - 1],
        ],
        dtype=np.float32,
    )

    homography = cv2.getPerspectiveTransform(
        document_corners.astype(np.float32),
        destination_corners,
    )

    return cv2.warpPerspective(
        image,
        homography,
        (output_width, output_height),
    )

Now we can simply do:
```python
canonical_image = rectify_document(
    photo_bgr,
    document_corners,
)
```

### Coordinate Scaling
This is the missing bridge between template-space coordinates and our new canonical space

In [ ]:
def scale_field_shape(
    shape,
    source_width,
    source_height,
    target_width=CANONICAL_WIDTH,
    target_height=CANONICAL_HEIGHT,
):
    scale_x = target_width / source_width
    scale_y = target_height / source_height

    return {
        "x": round(shape["x"] * scale_x),
        "y": round(shape["y"] * scale_y),
        "width": round(shape["width"] * scale_x),
        "height": round(shape["height"] * scale_y),
    }

### Crop Field
Turn the padding experiment into a reusable component:

In [ ]:
def crop_field(
    image,
    shape,
    padding_x=FIELD_PADDING_X,
    min_padding_x=FIELD_MIN_PADDING_X,
    padding_top=FIELD_PADDING_TOP,
    padding_bottom=FIELD_PADDING_BOTTOM,
):
    x = shape["x"]
    y = shape["y"]
    width = shape["width"]
    height = shape["height"]

    image_height, image_width = image.shape[:2]

    pad_x = max(round(width * padding_x), min_padding_x)
    pad_top = round(height * padding_top)
    pad_bottom = round(height * padding_bottom)

    x1 = max(0, x - pad_x)
    y1 = max(0, y - pad_top)

    x2 = min(
        image_width,
        x + width + pad_x,
    )

    y2 = min(
        image_height,
        y + height + pad_bottom,
    )

    return image[y1:y2, x1:x2]

Now one helper for all field crops:

In [ ]:
def extract_field_crops(
    canonical_image,
    template_record,
    template_width,
    template_height,
    field_names=REQUIRED_FIELDS,
):
    crops = {}

    for field_name in field_names:
        region = find_field_region(
            template_record,
            field_name,
        )

        canonical_shape = scale_field_shape(
            region["shape_attributes"],
            template_width,
            template_height,
        )

        crops[field_name] = crop_field(
            canonical_image,
            canonical_shape,
        )

    return crops

The final crop helper keeps at least 12 horizontal pixels of context. The earlier five-identity gallery exposed clipped `gender` characters with percentage padding alone. OpenCV uses BGR, so convert once to RGB before handing the canonical image and its crops to either extraction pipeline.

In [ ]:
photo_record = find_record(photo_metadata, SAMPLE_ID)
template_record = find_record(template_metadata, SAMPLE_ID)
photo_bgr = cv2.imread(str(photo_image_path))
template_bgr = cv2.imread(str(template_image_path))
assert photo_bgr is not None and template_bgr is not None

canonical_bgr = rectify_document(photo_bgr, extract_document_corners(photo_record))
canonical_rgb = cv2.cvtColor(canonical_bgr, cv2.COLOR_BGR2RGB)
clean_template_rgb = cv2.cvtColor(
    cv2.resize(template_bgr, (CANONICAL_WIDTH, CANONICAL_HEIGHT), interpolation=cv2.INTER_AREA),
    cv2.COLOR_BGR2RGB,
)
field_crops_rgb = extract_field_crops(
    canonical_rgb, template_record, template_bgr.shape[1], template_bgr.shape[0]
)
assert canonical_rgb.shape == clean_template_rgb.shape == (CANONICAL_HEIGHT, CANONICAL_WIDTH, 3)
assert set(field_crops_rgb) == set(REQUIRED_FIELDS)
assert all(crop.size > 0 for crop in field_crops_rgb.values())
print(canonical_rgb.shape, {field: crop.shape for field, crop in field_crops_rgb.items()})